# Libriaries

In [1]:
import numpy as np
import glob
import os
from array import array
import ROOT
import re

# Function definition

In [2]:
#Function for linear fitting
def linear(x_data, m, q):
    return x_data*m + q

# Function for crossing point computation
def intercept_lin(m1, q1, m2, q2):
    intercept_x = (q2-q1)/(m1-m2)
    return intercept_x

def ordina_wafer_sensore(name):
    match = re.search(r'W(\d+).*?S(\d+).*?LGAD-([A-Z])', name)
    if match:
        return (int(match.group(1)), int(match.group(2)),str(match.group(3)))
    else:
        return (9999, 9999, "Z")

# Data folder

In [3]:
folder = "Dati/ETL-PreSerie/LF-FBK/QC-TS/Batch1/CV/NoIrr" # Write here the path in which data are stored
pattern = "*.txt" 

path = os.path.join(folder, pattern)
file_list= glob.glob(path)

file_list.sort(key=ordina_wafer_sensore)

# Vgl Histogram

## Bulk fit range definition

In [4]:
p_low= 0.10
p_high= 0.75

## Linear fits and Vgl computing

In [5]:
# If you want to ignore warning about png files creation
ROOT.gErrorIgnoreLevel = ROOT.kWarning

#I create the Canvas on which I will sequentially plot the graphs and fits for all the TS
canvas = ROOT.TCanvas("canvas", "1/C^2[V]", 800, 600)

#Creation of the text file
with open("V_GL_results_LS-FBK.txt", "w") as f_out:
    f_out.write(
        "Wafer\tSensor\tLGAD\tm_GL\tq_GL\tm_bulk\tq_bulk\t"
        "V_GL\tN_punti_fit_bulk\tchi2_ridotto_bulk\tRMSE_bulk\n"
    )
for filename in file_list:
    #print(f"Elaboration of: {filename}")

    #Since some of the mesurements have no LGAD indicated:
    control= re.search(r'LGAD-([A-Z])',filename)
    if control:
        LGAD = control.group(1)
    else:
        LGAD = "?"

        
    # W-S-LGAD extraction
    match = re.search(r'W(\d+).*?S(\d+)', filename)        
    if match:
        wafer = match.group(1)
        sensor = match.group(2)
    else:
        wafer = "NA" 
        sensor = "NA"
        LGAD = "NA"


    canvas.Clear()

    with open(filename, "r") as file:
        Filelines = file.readlines()
        
    try:
        #start/end tension, voltage step and number of steps
        Start_Voltage = float(Filelines[22].split()[1])
        Stop_Voltage = float(Filelines[23].split()[1])
        Step_Voltage = abs(float(Filelines[25].split()[1]))
        Num_biasStep = int((-Stop_Voltage+Start_Voltage)/Step_Voltage)
        
        #print("\nBias ramp:")
        #print("Start = " + str(Start_Voltage) + " V")
        #print("Stop= " + str(Stop_Voltage) + " V")
        #print("Step= " + str(Step_Voltage) + " V")
        
        #List filling 
        Bias_Voltage = []
        Capacitance = []
        Capacitance2 = [] #1/C^2
        Line_first_data = 135 
        i = 0
      
        while i <= Num_biasStep:
            Bias_Voltage.append(abs(float(Filelines[Line_first_data + i].split()[0])))
            Capacitance.append(float(Filelines[Line_first_data + i].split()[1]))
            Capacitance2.append(1/(float(Filelines[Line_first_data + i].split()[1]))**2)
            i = i + 1
    
        # Auto-range system
        C2_low = min(Capacitance2) + (max(Capacitance2)-min(Capacitance2))* p_low
        C2_high = min(Capacitance2) + (max(Capacitance2)-min(Capacitance2))* p_high
        closest_low = min(Capacitance2, key=lambda x: abs(x - C2_low))
        closest_high = min(Capacitance2, key=lambda x: abs(x - C2_high))
        #print(closest_low,closest_high)
    
        lower_C2_index = Capacitance2.index(closest_low)
        higher_C2_index = Capacitance2.index(closest_high)
    
        lower_V= Bias_Voltage[lower_C2_index]
        higher_V=Bias_Voltage[higher_C2_index]
    
        ### FIT gain layer region ###
   
        low_GL_bias = 10.     # Fit lower voltage limit
        high_GL_bias = 30.    # Fit higher voltage limit 
        i_low_GL = int(low_GL_bias/Step_Voltage)   #lower index
        i_high_GL = int(high_GL_bias/Step_Voltage)    #higher index
        i_low_bulk = lower_C2_index   
        i_high_bulk = higher_C2_index  
        #print(i_low_bulk,i_high_bulk)
        
        Bias_Voltage_GL = Bias_Voltage[i_low_GL:i_high_GL]
        Capacitance2_GL = Capacitance2[i_low_GL:i_high_GL]
        Bias_Voltage_bulk = Bias_Voltage[i_low_bulk:i_high_bulk+1]
        Capacitance2_bulk = Capacitance2[i_low_bulk:i_high_bulk+1] 
            
        #Graph creation
        x_Data = array('d', Bias_Voltage)
        y_Data = array('d', Capacitance2)
    
        graph_Data = ROOT.TGraph(len(x_Data), x_Data, y_Data)
        graph_Data.SetMarkerStyle(20)      
        graph_Data.SetMarkerSize(0.5)
        graph_Data.SetMarkerColor(ROOT.kBlack)
        
        #GL fit
        fit_func_GL = ROOT.TF1("fit_func_GL", "[0]+[1]*x",Bias_Voltage_GL[0],Bias_Voltage_GL[-1]) 
        fit_func_GL.SetLineColor(ROOT.kRed)
        fit_func_GL.SetLineWidth(2)
        fit_func_GL.SetParameter(0,10**18)
        fit_func_GL.SetParameter(1,10**18)
    
        fit_GL = graph_Data.Fit(fit_func_GL, "RSQ") #R imposta il range relativo al fit, S salva e Q è la modalità quiet
        q_GL = fit_func_GL.GetParameter(0)
        m_GL = fit_func_GL.GetParameter(1)    

        
        # Bulk fit
        fit_func_bulk = ROOT.TF1("fit_func_bulk", "[0]+[1]*x",Bias_Voltage_bulk[0],Bias_Voltage_bulk[-1])
        fit_func_bulk.SetLineColor(ROOT.kGreen)
        fit_func_bulk.SetParameter(1,10**21)
        fit_func_bulk.SetParameter(0,-10**23)

        fit_bulk = graph_Data.Fit(fit_func_bulk, "RSQ+") #R imposta il range relativo al fit, S salva e Q è la modalità quiet, il + lo aggiunge allo stesso grafico
        q_bulk = fit_func_bulk.GetParameter(0)
        m_bulk = fit_func_bulk.GetParameter(1)
        
        # Results output
        #print("\nGain Layer region *** Linear fit")
        #print("Angular coefficinet: m = " + str(m_GL))
        #print("y-intercept: q = " + str(q_GL))
        Covariance_matrix_GL = fit_GL.GetCovarianceMatrix()
        #print("sigma_m = " + str(Covariance_matrix_GL(1,1)))
        #print("sigma_q = " + str(Covariance_matrix_GL(0,0)))
    
        #print("\nBulk region *** Linear fit")
        #print(f"Il range di fitting della regione di bulk è: {Bias_Voltage_bulk[0]} - {Bias_Voltage_bulk[-1]}  ->  {len(Bias_Voltage_bulk)} punti")
        #print("Angular coefficinet: m = " + str(m_bulk))
        #print("y-intercept: q = " + str(q_bulk))
        Covariance_matrix_bulk = fit_bulk.GetCovarianceMatrix()
        #print("sigma_m = " + str(Covariance_matrix_bulk(1,1)))
        #print("sigma_q = " + str(Covariance_matrix_bulk(0,0)))
    
        #print("\n    ***   Depletion voltage of the gain layer   ***")
        V_GL=intercept_lin(m_GL, q_GL, m_bulk, q_bulk)
        #print("V_GL = " + str(V_GL) + " V\n\n")

        
        
        #Plot
        
        # Range extension of the lines
        fit_func_GL.SetRange(Bias_Voltage[0], V_GL + 10)
        fit_func_bulk.SetRange(V_GL - 10, Bias_Voltage[-1])
        fit_func_GL.SetLineWidth(3)
        fit_func_bulk.SetLineWidth(1)
        
        graph_Data.SetTitle(f"Sensor W{wafer} S{sensor} \nV_GL = {V_GL:.2f} V ; Bias Voltage [V];1/C^{2} [1/F^{2}]")

        graph_Data.Draw("AP")
        graph_Data.SetMinimum(-10*10**21) 
        fit_func_GL.Draw("SAME")
        fit_func_bulk.Draw("SAME")
    
        canvas.Draw()
        canvas.Update()
    
        low_line = ROOT.TLine(0, closest_low, 60,closest_low)
        low_line.SetLineColor(ROOT.kRed)
        low_line.SetLineStyle(2)
        #low_line.SetLineWidth(0.7)
        low_line.Draw() 
    
        high_line = ROOT.TLine(0, closest_high, 60,closest_high)
        high_line.SetLineColor(ROOT.kRed)
        high_line.SetLineStyle(2)
        #high_line.SetLineWidth(0.7)
        high_line.Draw()

        ROOT.gStyle.SetOptStat(0) 
        ROOT.gStyle.SetOptFit(0)   

        # Canvas legend
        legend = ROOT.TLegend(0.10, 0.75, 0.55, 0.88)
        legend.SetBorderSize(1)
        legend.SetFillColor(0)
        legend.SetTextSize(0.025)
        
        chi2_bulk = fit_func_bulk.GetChisquare()
        n_punti_bulk = len(Bias_Voltage_bulk)
        ndf_bulk = fit_func_bulk.GetNDF()
        
        chi2_red_bulk = chi2_bulk / ndf_bulk


        # RMSE computation -  bulk fit
        residui_bulk = [
        y - (q_bulk + m_bulk * x)
        for x, y in zip(Bias_Voltage_bulk, Capacitance2_bulk)
        ]

        rmse_bulk = np.sqrt(np.mean(np.array(residui_bulk)**2))
        



        legend.AddEntry(
            fit_func_bulk,
            f"Bulk fit: #chi^{{2}}/NDF = {(chi2_red_bulk/10**40):.2f}, "
            f"N = {n_punti_bulk}, NDF = {ndf_bulk}",
            "l"
        )
        
        legend.Draw()
        
        canvas.Draw()
        png_name = filename.replace(".txt", ".png")
        canvas.SaveAs(png_name)


    
        
        # Saving the  results in a txt text
        with open("V_GL_results_LS-FBK.txt", "a") as f_out:
            f_out.write(
            f"{wafer}\t{sensor}\t{LGAD}\t{m_GL}\t{q_GL}\t"
            f"{m_bulk}\t{q_bulk}\t{V_GL}\t{n_punti_bulk}\t"
            f"{chi2_red_bulk}\t{rmse_bulk}\n")
        
    
                
    except (IndexError, ValueError) as e:
        print(f" -> Errore di formattazione nel file {filename}: {e}")


 -> Errore di formattazione nel file Dati/ETL-PreSerie/LF-FBK/QC-TS/Batch1/CV/NoIrr/index.txt: could not convert string to float: 'Sweep'


# Vgl Histogram

## Number of bins

In [6]:
nbins= 40

In [7]:
results = "V_GL_results_LS-FBK.txt"
V_GL_values = []
N_points= [] #number of points of the bulk fit
Chi2=[]
RMSE=[]

with open(results, "r") as f:
    lines = f.readlines()
    for line in lines[1:]:
        Values = line.strip().split('\t')
        vgl_value = float(Values[7])
        V_GL_values.append(vgl_value)
        n_points= float(Values[8])
        N_points.append(n_points)
        chi2 = float(Values[9])
        chi2_scaled= float(chi2/10**40)
        Chi2.append(chi2_scaled)
        rmse=float(Values[10])
        RMSE.append(rmse)

#Graph creation
ROOT.gStyle.SetOptStat(1111)
ROOT.gStyle.SetOptFit(111)
ROOT.gStyle.SetStatX(0.88)
ROOT.gStyle.SetStatY(0.85)
ROOT.gStyle.SetStatW(0.20)
ROOT.gStyle.SetStatH(0.15)

canvas_hist = ROOT.TCanvas("canvas_hist", "Distribuzione V_GL", 800, 600)

hist_V_GL = ROOT.TH1F("hist_V_GL", "Distribuzione V_{GL};V_{GL} [V];Sensori",nbins,45 ,55 )
hist_V_GL.SetFillColor(ROOT.kAzure+1)
hist_V_GL.SetLineColor(ROOT.kBlack)

for val in V_GL_values:
    hist_V_GL.Fill(val)

hist_V_GL.Fit("gaus", "Q") 

# Mean and Sigma  extraction
fit_func = hist_V_GL.GetFunction("gaus")

media = fit_func.GetParameter(1)
sigma = fit_func.GetParameter(2)
chi2 = fit_func.GetChisquare()
ndf = fit_func.GetNDF()
chi2_red = chi2 / ndf

print(f"\nRisultati del Fit Gaussiano:")
print(f"Media (V_GL) = {media:.2f} V")
print(f"Sigma = {sigma:.2f} V")
print(f"Chi^2 Ridotto = {chi2_red:.2f}")

mean_N_points = np.mean(N_points)
mean_Chi2=np.mean(Chi2)
mean_RMSE=np.mean(RMSE)

print("\n---- FIT NELLA REGIONE DI BULK ------------------------")
print(f"N medio di punti fit bulk = {mean_N_points:.2f}; min = {min(N_points)}  ; max = {max(N_points)}")
print(f"Chi^2 ridotto medio per fit bulk = {mean_Chi2:.4f}  ; min = {min(Chi2)}  ; max = {max(Chi2)}")
print(f"RMSE medio = {mean_RMSE/(10**21):.4f} * 10^21  ; min = {min(RMSE)/(10**21):.4f} * 10^21  ; max = {max(RMSE)/(10**21):.4f}* 10^21")

#legend = ROOT.TLegend(0.60, 0.30, 0.88, 0.50)
#legend.SetBorderSize(1)
#legend.SetFillColor(0)
#legend.SetTextSize(0.035)
#legend.SetMargin(0.05)

#legend.AddEntry("", f"N_green_fit = {mean_N_points:.2f}", "")

hist_V_GL.Draw()
#legend.Draw()
canvas_hist.Draw()



canvas_hist.SaveAs("Istogramma_V_GL_LS-FBK.png")



Risultati del Fit Gaussiano:
Media (V_GL) = 48.78 V
Sigma = 0.36 V
Chi^2 Ridotto = 1.00

---- FIT NELLA REGIONE DI BULK ------------------------
N medio di punti fit bulk = 3.49; min = 3.0  ; max = 5.0
Chi^2 ridotto medio per fit bulk = 4981.8422  ; min = 17.401306436580196  ; max = 24827.12181898984
RMSE medio = 3.8568 * 10^21  ; min = 0.2408 * 10^21  ; max = 9.0971* 10^21


# Vgl differences Histogram

## Number of bins

In [8]:
#Bins for differences histogram
nbins = 40

## Creation of On_wafer_data Dictionary

In [9]:
from striprtf.striprtf import rtf_to_text

On_wafer_data={}

with open("FBK_OnWafer_Vgl.txt", "r") as file:
    
    content = file.read()
    lines = content.split('\n')[1::]
    for line in lines:
        On_wafer_VGL=float(line.split("\t")[1])
        On_wafer_W=line.split("_")[3].split("W")[1]
        On_wafer_S=line.split("_")[4].split("\t")[0].split("S")[1]
        On_wafer_data[(On_wafer_W,On_wafer_S)] = On_wafer_VGL
print(len(On_wafer_data))


72


## Creation of data differences dictionary

In [10]:
diff_data={}

with open("V_GL_results_LS-FBK.txt","r") as file:
    lines= file.read().split("\n")[1::]
    for line in lines:
        if len(line.split("\t"))== 11:
            VGL_result=float(line.split("\t")[7])
            W_result=line.split("\t")[0]
            S_result=line.split("\t")[1]

            
            #Calcolo le differenze fra le V_gl
            if (W_result,S_result) in On_wafer_data:
                diff_data[(W_result, S_result)]= float(On_wafer_data[(W_result,S_result)] - VGL_result)

## Plotting

In [11]:
ROOT.gStyle.SetOptStat(1111)
ROOT.gStyle.SetOptFit(111)
ROOT.gStyle.SetStatX(0.88)
ROOT.gStyle.SetStatY(0.85)
ROOT.gStyle.SetStatW(0.20)
ROOT.gStyle.SetStatH(0.15)

canvas_diff = ROOT.TCanvas("canvas_diff", "Distribuzione Differenze V_GL", 800, 600)

diff_values = list(diff_data.values())

#Margini istogramma
min_diff = min(diff_values)
max_diff = max(diff_values)
margin = 0.5 #oppure :(max_diff - min_diff) * 0.2 if max_diff != min_diff else 0.5
# Creazione dell'istogramma
hist_diff = ROOT.TH1F("hist_diff", "Differenza V_{GL} (On Wafer - Risultato Fit);#DeltaV_{GL} [V];Sensori", nbins, min_diff - margin, max_diff + margin)
hist_diff.SetFillColor(ROOT.kAzure+1)
hist_diff.SetLineColor(ROOT.kBlack)

# Riempimento istogramma
for val in diff_values:
    hist_diff.Fill(val)

# Fit gaussiano
hist_diff.Fit("gaus", "Q")

# Estrazione e stampa dei parametri dal fit
fit_func = hist_diff.GetFunction("gaus")

if fit_func:
    mean = fit_func.GetParameter(1)
    sigma = fit_func.GetParameter(2)
    chi2 = fit_func.GetChisquare()
    ndf = fit_func.GetNDF()
    chi2_red = chi2 / ndf if ndf != 0 else 0

    print(f"\n--- Risultati del Fit Gaussiano sulle Differenze ---")
    print(f"Media (#DeltaV_GL) = {mean:.4f} V")
    print(f"Sigma = {sigma:.4f} V")
    print(f"Chi^2 Ridotto = {chi2_red:.2f}")

hist_diff.Draw()
canvas_diff.Draw()
canvas_diff.SaveAs("Istogramma_Differenze_V_GL_LS-FBK.png")


--- Risultati del Fit Gaussiano sulle Differenze ---
Media (#DeltaV_GL) = -0.0292 V
Sigma = 0.0670 V
Chi^2 Ridotto = 0.92
